### Задача 2 Нелинейное ДП: Оптимизация распределения нагрузки

In [2]:
import numpy as np
from scipy.optimize import linprog, minimize_scalar
from scipy.interpolate import interp1d

alpha, beta, N, X0 = 0.7, 0.3, 3, 6
grid = np.arange(0, 7, 1.0)
cost = lambda Y, X: Y ** 2 + 2 * (X - Y) ** 2
F, Yopt = {}, {}
 
F[1] = np.zeros_like(grid)
Yopt[1] = np.zeros_like(grid)

for i, X in enumerate(grid):
    if X == 0: 
        continue
    r = minimize_scalar(lambda Y: cost(Y, X), bounds=(0, X), method='bounded')
    F[1][i], Yopt[1][i] = r.fun, r.x
 
for k in range(2, N + 1):
    F[k] = np.zeros_like(grid); Yopt[k] = np.zeros_like(grid)
    fprev = interp1d(grid, F[k - 1], kind='linear', fill_value='extrapolate')
    for i, X in enumerate(grid):
        if X == 0:
            continue
        tot = lambda Y: cost(Y, X) + float(fprev(alpha * Y + beta * (X - Y)))
        r = minimize_scalar(tot, bounds=(0, X), method='bounded')
        F[k][i], Yopt[k][i] = r.fun, r.x
 
Xc, total = X0, 0.0
for k in range(N, 0, -1):
    y = float(interp1d(grid, Yopt[k])(Xc))
    y = max(0, min(y, Xc))
    total += cost(y, Xc)
    Xc = alpha * y + beta * (Xc - y)

print(f'Суммарные затраты = {round(total, 3)}')


Суммарные затраты = 33.649
